# IOAI — 2025 Selection Road Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_file('benin-national-ai-olympiad-selection', 'train.csv', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Benin — National AI Olympiad Selection 2025 (Road Detection)

> **Bénin National Olympiad in AI — Selection round · Kaggle playground competition**

위성/항공 이미지에 **도로가 있는지(`target` 0/1)** 예측합니다. 입력은 이미지에서 미리 뽑아 익명화한
**285개 특징 `f0..f284`** 입니다(Zindi 원문제는 도로 이미지 분류, 캐글판은 그 특징 벡터로 제공). 지표는
**ROC-AUC**(높을수록 좋음) — 임계값을 거치지 않은 **확률**을 제출합니다.

공식 Kaggle test 라벨은 비공개라, 여기서는 `train.csv` 의 **결정적 held-out 분할(`id % 5 == 0`)**로 채점합니다.
**제출 파일** `submission.csv` (`id,target`) 를 만들면 사이트 **Submissions** 탭에서 AUC 채점됩니다.

⚠️ **아래 베이스라인은 285개 특징을 그대로 HistGradientBoosting 에 넣은 단순 모델(~0.84 AUC)** 입니다.
특징 전처리·앙상블·하이퍼파라미터 튜닝(모범답안 참고)으로 AUC 를 더 끌어올리세요.


In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data/train.csv')
FEATS = [f'f{i}' for i in range(285)]   # f0..f284 (익명 특징)
is_val = df['id'] % 5 == 0                      # 결정적 held-out (채점과 동일)
tr, va = df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)
print('train', len(tr), 'val', len(va), '| target mean', round(df['target'].mean(), 4))


In [ ]:
# BASELINE: 285개 특징 전체를 HistGradientBoosting 에 그대로
from sklearn.ensemble import HistGradientBoostingClassifier
clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=0)
clf.fit(tr[FEATS], tr['target'])
proba = clf.predict_proba(va[FEATS])[:, 1]
print('held-out ROC-AUC:', round(roc_auc_score(va['target'], proba), 4))

pd.DataFrame({'id': va['id'], 'target': proba}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(va), '(특징 전체 baseline — 전처리/앙상블로 개선하세요)')


### (선택) 실제 캐글 리더보드 제출용
아래는 공식 `test.csv` 에 대한 예측을 `submission_test.csv` 로 저장합니다(캐글에 직접 업로드용).
사이트 채점에는 위 `submission.csv`(held-out) 만 사용되므로 이 셀은 건너뛰어도 됩니다.

In [ ]:
import os
if os.path.exists('data/test.csv'):
    te = pd.read_csv('data/test.csv')
    tp = clf.predict_proba(te[FEATS])[:, 1]
    pd.DataFrame({'id': te['id'], 'target': tp}).to_csv('submission_test.csv', index=False)
    print('wrote submission_test.csv', len(te))
else:
    print('data/test.csv 없음 — 캐글 제출용은 생략(사이트 채점은 submission.csv 로 충분)')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)